In [1]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler, OneHotEncoder, OrdinalEncoder
from sklearn.pipeline import Pipeline
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

from src.data import load_data, get_column_types, TARGET, RISK_ORDER, EXPECTED_CATEGORIES
from src.preprocessing import prepare_data, RANDOM_STATE

In [2]:
df_raw = load_data()
df = df_raw.copy()

for col in df.select_dtypes("object"):
    df[col] = df[col].str.strip()

print("Duplicates:", df.duplicated().sum())
df = df.drop_duplicates()

for col, values in EXPECTED_CATEGORIES.items():
    invalid = ~df[col].isin(values) & df[col].notna()
    print(col, "invalid:", invalid.sum())
    df.loc[invalid, col] = None

inconsistent = df["Screen_Time_Hours"] < df["Social_Media_Hours"]
print("Screen_Time < Social_Media:", inconsistent.sum())
df = df[~inconsistent]

print("Missing target:", df[TARGET].isna().sum())
df = df.dropna(subset=[TARGET]).reset_index(drop=True)
print("Rows:", len(df_raw), "->", len(df))

Duplicates: 0
Gender invalid: 0
Physical_Activity invalid: 0
Part_Time_Job invalid: 0
Diet_Quality invalid: 0
Previous_Mental_Health_History invalid: 0
Mental_Health_Risk invalid: 0
Screen_Time < Social_Media: 37
Missing target: 0
Rows: 2000 -> 1963


C:\Users\USER\AppData\Local\Temp\ipykernel_10780\1921603204.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes("object"):


In [3]:
X = df.drop(columns=[TARGET, "Student_ID"])
y = df[TARGET].map({r: i for i, r in enumerate(RISK_ORDER)})

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print("Train:", X_train.shape, "| Test:", X_test.shape)
print("Train classes:\n", y_train.value_counts(normalize=True).round(2))
print("Test classes:\n", y_test.value_counts(normalize=True).round(2))

Xtr, Xte, ytr, yte = prepare_data(df_raw)
assert Xtr.equals(X_train) and Xte.equals(X_test) and ytr.equals(y_train) and yte.equals(y_test)

Train: (1570, 15) | Test: (393, 15)
Train classes:
 Mental_Health_Risk
2    0.34
0    0.33
1    0.33
Name: proportion, dtype: float64
Test classes:
 Mental_Health_Risk
2    0.34
0    0.33
1    0.33
Name: proportion, dtype: float64


In [4]:
def missing(d):
    return pd.DataFrame({"missing values": d.isna().sum(), "percentage": d.isna().mean() * 100}).query("`missing values` > 0")

print("Train missing values:\n", missing(X_train))
print("Test missing values:\n", missing(X_test))

Train missing values:
                     missing values  percentage
Sleep_Hours                     48    3.057325
GPA                             29    1.847134
Social_Media_Hours              46    2.929936
Financial_Stress                72    4.585987
Family_Support                  43    2.738854
Attendance_Rate                 40    2.547771
Diet_Quality                    52    3.312102
Test missing values:
                     missing values  percentage
Sleep_Hours                     11    2.798982
GPA                              8    2.035623
Social_Media_Hours              12    3.053435
Financial_Stress                19    4.834606
Family_Support                  11    2.798982
Attendance_Rate                  6    1.526718
Diet_Quality                    15    3.816794


In [5]:
numeric_cols, _ = get_column_types(X_train)

rows = []
for col in numeric_cols:
    q1, q3 = X_train[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    rows.append([col, lo, hi, X_train[col].min(), X_train[col].max(),
                 ((X_train[col] < lo) | (X_train[col] > hi)).sum()])
pd.DataFrame(rows, columns=["column", "lower", "upper", "min", "max", "n_outliers"])

,column,lower,upper,min,max,n_outliers
0,Age,15.50,27.50,17.0,28.00,3
1,Year_of_Study,-3.50,8.50,1.0,5.00,0
2,Sleep_Hours,3.75,9.75,3.3,10.00,9
3,Study_Hours_Per_Week,4.10,35.90,2.4,38.30,7
4,GPA,1.64,3.48,1.5,3.66,7
5,Social_Media_Hours,3.20,8.80,2.1,10.00,21
6,Screen_Time_Hours,4.20,13.00,3.4,14.00,8
7,Financial_Stress,0.25,11.85,1.5,9.90,0
8,Family_Support,1.00,12.20,1.7,9.90,0
9,Attendance_Rate,65.50,104.70,54.4,100.00,11


In [6]:
counts = y_train.value_counts()
ratio = counts.max() / counts.min()
USE_SMOTE = ratio > 1.5

print("Class counts:", counts.to_dict())
print("Imbalance ratio:", round(ratio, 2))
print("USE_SMOTE:", USE_SMOTE)

Class counts: {2: 536, 0: 518, 1: 516}
Imbalance ratio: 1.04
USE_SMOTE: False


In [7]:
ORDINAL_COLS = {
    "Physical_Activity": ["Low", "Moderate", "High"],
    "Diet_Quality": ["Poor", "Average", "Good"],
}

numeric, categorical = get_column_types(X_train)
ordinal = list(ORDINAL_COLS)
nominal = [c for c in categorical if c not in ordinal]

preprocessor = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")),
                      ("scaler", RobustScaler())]), numeric),
    ("ord", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                      ("encoder", OrdinalEncoder(categories=list(ORDINAL_COLS.values())))]), ordinal),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                      ("encoder", OneHotEncoder(handle_unknown="ignore"))]), nominal),
])

X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep = preprocessor.transform(X_test)

print("Processed train:", X_train_prep.shape)
print("Processed test:", X_test_prep.shape)
print("NaN values:", np.isnan(X_train_prep).sum(), np.isnan(X_test_prep).sum())

Processed train: (1570, 19)
Processed test: (393, 19)
NaN values: 0 0


In [8]:
# leakage check: statistics come from train only
print("Median used (Sleep_Hours): train =", X_train["Sleep_Hours"].median(), "| test =", X_test["Sleep_Hours"].median())
print("Imputer fitted on:", preprocessor.named_transformers_["num"]["imputer"].statistics_[:3])

# full pipeline layout (model plugged in at step 3)
steps = [("preprocessing", preprocessor)]
if USE_SMOTE:
    steps.append(("smote", SMOTE(random_state=RANDOM_STATE)))
steps

Median used (Sleep_Hours): train = 6.8 | test = 6.7
Imputer fitted on: [21.   3.   6.8]


[('preprocessing',
  ColumnTransformer(transformers=[('num',
                                   Pipeline(steps=[('imputer',
                                                    SimpleImputer(strategy='median')),
                                                   ('scaler', RobustScaler())]),
                                   ['Age', 'Year_of_Study', 'Sleep_Hours',
                                    'Study_Hours_Per_Week', 'GPA',
                                    'Social_Media_Hours', 'Screen_Time_Hours',
                                    'Financial_Stress', 'Family_Support',
                                    'Attendance_Rate']),
                                  ('ord',
                                   Pipeline(steps=[('imputer',
                                                    SimpleImputer(strategy='most_frequent')),
                                                   ('encoder',
                                                    OrdinalEncoder(categories=[['Low',
       